In [2]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(
    feature_data["date"]
)

feature_data.head()

,Id,date,TotalSteps,Calories,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,recorded_sleep_minutes,baseline_steps_7d,...,steps_deviation_7d,steps_deviation_7d_pct,sleep_deviation_7d,sleep_deviation_7d_pct,steps_rolling_3d,sleep_rolling_3d,activity_ratio,sleep_ratio,day_number,is_baseline
0,1503960366,2016-03-25,11004,1819,33,12,205,804,424,13019.857143,...,-2015.857143,-15.482944,44.285714,11.662904,11004.000000,424.000000,0.845171,1.116629,1,True
1,1503960366,2016-03-26,17609,2154,89,17,274,588,511,13019.857143,...,4589.142857,35.247260,131.285714,34.574868,14306.500000,467.500000,1.352473,1.345749,2,True
2,1503960366,2016-03-27,12736,1944,56,5,268,605,533,13019.857143,...,-283.857143,-2.180186,153.285714,40.368698,13783.000000,489.333333,0.978198,1.403687,3,True
3,1503960366,2016-03-28,13231,1932,39,20,224,1080,81,13019.857143,...,211.142857,1.621699,-298.714286,-78.668172,14525.333333,375.000000,1.016217,0.213318,4,True
4,1503960366,2016-03-29,12041,1886,28,28,243,763,411,13019.857143,...,-978.857143,-7.518187,31.285714,8.239278,12669.333333,341.666667,0.924818,1.082393,5,True


In [3]:
# Evaluation observations: first 7 observations ke baad
evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

print("Evaluation observations:", evaluation_data.shape[0])

print("\nSteps deviation (%)")
print(evaluation_data["steps_deviation_7d_pct"].describe())

print("\nSleep deviation (%)")
print(evaluation_data["sleep_deviation_7d_pct"].describe())

Evaluation observations: 74

Steps deviation (%)
count    74.000000
mean    -16.317047
std      37.882650
min     -99.220237
25%     -37.768842
50%      -9.295157
75%       6.851604
max      67.628397
Name: steps_deviation_7d_pct, dtype: float64

Sleep deviation (%)
count    74.000000
mean     -4.110874
std      31.778585
min     -95.549339
25%     -19.816511
50%      -2.001602
75%      14.084502
max      68.445975
Name: sleep_deviation_7d_pct, dtype: float64


In [4]:
# Sort observations chronologically for each participant
evaluation_data = evaluation_data.sort_values(
    ["Id", "day_number"]
).copy()

# Identify deviations of 10% or more below personal baseline
evaluation_data["steps_low"] = (
    evaluation_data["steps_deviation_7d_pct"] <= -10
)

evaluation_data["sleep_low"] = (
    evaluation_data["sleep_deviation_7d_pct"] <= -10
)

# Flag 4 consecutive available observations
evaluation_data["steps_4_obs_alert"] = (
    evaluation_data.groupby("Id")["steps_low"]
    .transform(lambda s: s.rolling(4, min_periods=4).sum().eq(4))
)

evaluation_data["sleep_4_obs_alert"] = (
    evaluation_data.groupby("Id")["sleep_low"]
    .transform(lambda s: s.rolling(4, min_periods=4).sum().eq(4))
)

print("Steps alert observations:",
      evaluation_data["steps_4_obs_alert"].sum())

print("Participants with steps alerts:",
      evaluation_data.loc[
          evaluation_data["steps_4_obs_alert"], "Id"
      ].nunique())

print("Sleep alert observations:",
      evaluation_data["sleep_4_obs_alert"].sum())

print("Participants with sleep alerts:",
      evaluation_data.loc[
          evaluation_data["sleep_4_obs_alert"], "Id"
      ].nunique())

Steps alert observations: 2
Participants with steps alerts: 1
Sleep alert observations: 1
Participants with sleep alerts: 1


In [5]:
# Combine activity and sleep deviation flags
evaluation_data["overall_deviation"] = (
    evaluation_data["steps_4_obs_alert"]
    | evaluation_data["sleep_4_obs_alert"]
)

print("Overall alert observations:",
      evaluation_data["overall_deviation"].sum())

print("Participants with any alert:",
      evaluation_data.loc[
          evaluation_data["overall_deviation"], "Id"
      ].nunique())

Overall alert observations: 3
Participants with any alert: 2
